# MVP v0.2.5.1: OPE with 90% Target Policy (200demos_epoch40)

**Date:** 2026-03-13  
**Builds on:** MVP v0.2.5 (reward model fix, episode-level binary reward)

## Goal

Re-run v0.2.5's OPE pipeline with a **different target policy** (200demos_epoch40, ~90% SR)
while keeping everything else identical:
- Same chunk diffuser (pre-trained from v0.2.5 on 54% SR target + expert data)
- Same behavior policy (BC_Gaussian trained on 54% SR target rollouts)
- Same normalization (computed from v0.2.5's data mix)

This tests whether SOPE can accurately estimate value for a stronger policy using the
same data distribution and diffuser.

## Changes from v0.2.5

1. **Target policy scorer**: 200demos_epoch40 (~90% SR) instead of test/20260309132349 (~54% SR)
2. **Oracle value**: 0.90 (vs 0.54 in v0.2.5)
3. **Chunk diffuser**: loaded from v0.2.5 checkpoint (no retraining)
4. **Behavior policy**: trained on same 54% SR rollout data as v0.2.5

## Pipeline

1. Load oracle value (pre-computed: V^pi = 0.90)
2. Load pre-collected rollouts (50 episodes from 54% SR policy) + expert demos (200 episodes)
3. Chunk all episodes into training data (for normalization only)
4. Load pre-trained chunk diffuser from v0.2.5
5. Train BC_Gaussian behavior policy on 54% SR rollout data (same as v0.2.5)
6. Load 200demos_epoch40 target policy scorer
7. Guided stitching with corrected SOPE hyperparameters
8. Evaluate OPE vs oracle (with fixed reward function)

In [1]:
import sys, os
import importlib
import numpy as np
import torch
import torch.nn as nn
import h5py
import json
import math
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from pathlib import Path
from collections import OrderedDict
from copy import deepcopy
from tqdm import tqdm
import time

# Project root
PROJECT_ROOT = Path("/home1/reishuen/latent_sope")
sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT / "src"))
sys.path.insert(0, str(PROJECT_ROOT / "third_party" / "sope"))
sys.path.insert(0, str(PROJECT_ROOT / "third_party" / "robomimic"))

# SOPE imports
from opelab.core.baselines.diffusion.temporal import TemporalUnet
from opelab.core.baselines.diffusion.diffusion import GaussianDiffusion
from opelab.core.baselines.diffusion.helpers import EMA, apply_conditioning

# Robomimic imports
import robomimic.utils.file_utils as FileUtils
import robomimic.utils.obs_utils as ObsUtils

# Our imports
import latent_sope.robomimic_interface.guidance as _guidance_mod
importlib.reload(_guidance_mod)
from latent_sope.robomimic_interface.checkpoints import (
    load_checkpoint, build_algo_from_checkpoint
)
from latent_sope.robomimic_interface.guidance import RobomimicDiffusionScorer

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

# Paths
DEMO_HDF5 = PROJECT_ROOT / "third_party/robomimic/datasets/lift/ph/low_dim_v15.hdf5"
# Original v0.2.5 checkpoint (for behavior data + normalization)
CKPT_DIR_V025 = PROJECT_ROOT / "third_party/robomimic/diffusion_policy_trained_models/test/20260309132349"
# New target policy: 200demos_epoch40 (~90% SR)
CKPT_DIR_TARGET = PROJECT_ROOT / "third_party/robomimic/diffusion_policy_trained_models/lift_diffusion_200demos/20260311141036"
RESULTS_DIR = PROJECT_ROOT / "results/2026-03-13"
TARGET_ROLLOUT_DIR = PROJECT_ROOT / "rollouts" / "target_policy_50"  # same rollout data as v0.2.5
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

CLIPTextModelWithProjection LOAD REPORT from: openai/clip-vit-large-patch14
Key                                                            | Status     |  | 
---------------------------------------------------------------+------------+--+-
vision_model.encoder.layers.{0...23}.self_attn.k_proj.bias     | UNEXPECTED |  | 
vision_model.encoder.layers.{0...23}.self_attn.v_proj.bias     | UNEXPECTED |  | 
vision_model.encoder.layers.{0...23}.self_attn.out_proj.bias   | UNEXPECTED |  | 
vision_model.encoder.layers.{0...23}.self_attn.q_proj.weight   | UNEXPECTED |  | 
vision_model.encoder.layers.{0...23}.self_attn.out_proj.weight | UNEXPECTED |  | 
vision_model.encoder.layers.{0...23}.layer_norm2.weight        | UNEXPECTED |  | 
vision_model.encoder.layers.{0...23}.layer_norm2.bias          | UNEXPECTED |  | 
vision_model.encoder.layers.{0...23}.self_attn.v_proj.weight   | UNEXPECTED |  | 
vision_model.encoder.layers.{0...23}.self_attn.q_proj.bias     | UNEXPECTED |  | 
vision_model.encoder.l

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


Device: cuda


## Configuration

In [2]:
# ── Obs keys (sorted, matching robomimic convention & LowDimConcatEncoder layout) ──
OBS_KEYS = ["object", "robot0_eef_pos", "robot0_eef_quat", "robot0_gripper_qpos"]

# Full dims
STATE_DIM = 19   # object(10) + eef_pos(3) + eef_quat(4) + gripper_qpos(2)
ACTION_DIM = 7   # full robomimic action space
TRANSITION_DIM = STATE_DIM + ACTION_DIM  # 26

# ── Target rollouts (same as v0.2.5) ──
NUM_TARGET_ROLLOUTS = 50   # pre-collected from March 9 (54% SR policy)

# ── Chunk config ──
CHUNK_SIZE = 4
FRAME_STACK = 1  # no frame stacking for low_dim_concat
STRIDE = 2

# ── Diffusion config (same architecture as v0.2.5) ──
N_DIFFUSION_STEPS = 256
DIM_MULTS = (1, 4, 8)
BASE_DIM = 32
ACTION_WEIGHT = 5.0
PREDICT_EPSILON = False  # x0-prediction (per D2.1 finding)

# ── Training config (for BC only — diffuser is pre-loaded) ──
TRAIN_EPOCHS = 50
BATCH_SIZE = 64
LR = 3e-4
GRAD_CLIP = 1.0

# ── Oracle config (200demos_epoch40, from oracle_eval_all_checkpoints.json) ──
ORACLE_VALUE = 0.90
ORACLE_SR = 0.90

# ── OPE config ──
NUM_SYNTHETIC_TRAJS = 50
T_GEN = 60
GAMMA = 1.0

# ── Reward ──
CUBE_Z_INDEX = 2  # index 2 in the 19-dim latent (object key, cube_pos z)
LIFT_THRESHOLD = 0.84

# ── Guidance config (matching SOPE diffusion policy config) ──
K_GUIDE = 1                # SOPE default for diffusion policies
NORMALIZE_GRAD = True      # SOPE default
USE_ADAPTIVE = False       # SOPE: false for diffusion policies
CLAMP = False              # SOPE: false for diffusion policies (critical fix!)
L_INF = 1.0                # unused when clamp=false

# Sweep: (action_scale, ratio) pairs
GUIDANCE_CONFIGS = [
    {"action_scale": 0.0,  "ratio": 0.0,  "label": "unguided"},
    # Positive-only baselines
    {"action_scale": 0.05, "ratio": 0.0,  "label": "pos_only_0.05"},
    {"action_scale": 0.1,  "ratio": 0.0,  "label": "pos_only_0.1"},
    {"action_scale": 0.2,  "ratio": 0.0,  "label": "pos_only_0.2"},
    # Full guidance: SOPE-like settings
    {"action_scale": 0.05, "ratio": 0.25, "label": "full_0.05_r0.25"},
    {"action_scale": 0.1,  "ratio": 0.25, "label": "full_0.1_r0.25"},
    {"action_scale": 0.2,  "ratio": 0.25, "label": "full_0.2_r0.25"},
    {"action_scale": 0.2,  "ratio": 0.5,  "label": "full_0.2_r0.5"},
    {"action_scale": 0.5,  "ratio": 0.25, "label": "full_0.5_r0.25"},
    {"action_scale": 0.5,  "ratio": 0.5,  "label": "full_0.5_r0.5"},
]

print(f"state_dim={STATE_DIM}, action_dim={ACTION_DIM}, transition_dim={TRANSITION_DIM}")
print(f"k_guide={K_GUIDE}, adaptive={USE_ADAPTIVE}, clamp={CLAMP}")
print(f"Target rollouts: {NUM_TARGET_ROLLOUTS} (54% SR, same as v0.2.5)")
print(f"Target policy for scoring: 200demos_epoch40 (~90% SR)")
print(f"Oracle V^pi = {ORACLE_VALUE}")
print(f"\nGuidance configs to sweep ({len(GUIDANCE_CONFIGS)}):")
for gc in GUIDANCE_CONFIGS:
    print(f"  {gc['label']}: action_scale={gc['action_scale']}, ratio={gc['ratio']}")

state_dim=19, action_dim=7, transition_dim=26
k_guide=1, adaptive=False, clamp=False
Target rollouts: 50 (54% SR, same as v0.2.5)
Target policy for scoring: 200demos_epoch40 (~90% SR)
Oracle V^pi = 0.9

Guidance configs to sweep (10):
  unguided: action_scale=0.0, ratio=0.0
  pos_only_0.05: action_scale=0.05, ratio=0.0
  pos_only_0.1: action_scale=0.1, ratio=0.0
  pos_only_0.2: action_scale=0.2, ratio=0.0
  full_0.05_r0.25: action_scale=0.05, ratio=0.25
  full_0.1_r0.25: action_scale=0.1, ratio=0.25
  full_0.2_r0.25: action_scale=0.2, ratio=0.25
  full_0.2_r0.5: action_scale=0.2, ratio=0.5
  full_0.5_r0.25: action_scale=0.5, ratio=0.25
  full_0.5_r0.5: action_scale=0.5, ratio=0.5


## Step 0: Load Oracle Values

In [3]:
# Oracle value pre-computed from oracle_eval_all_checkpoints.json (50 rollouts)
# 200demos_epoch40: mean_return=0.90, std=0.30, SR=90%
oracle_value = ORACLE_VALUE
oracle_success_rate = ORACLE_SR
oracle_std = 0.30

print(f"Oracle V^pi = {oracle_value:.4f} +/- {oracle_std:.4f}")
print(f"Oracle success rate: {oracle_success_rate*100:.1f}%")
print(f"(Source: oracle_eval_all_checkpoints.json, 50 rollouts, 200demos_epoch40)")

Oracle V^pi = 0.9000 +/- 0.3000
Oracle success rate: 90.0%
(Source: oracle_eval_all_checkpoints.json, 50 rollouts, 200demos_epoch40)


## Step 1: Load Pre-collected Target Policy Rollouts

Using the same 50 rollouts from v0.2.5 (54% SR policy, March 9).
These are stored at `rollouts/target_policy_50/` with format:
- `latents`: (T, 2, 19) — frame_stack=2, last frame extracted
- `actions`: (T, 7)
- `rewards`: (T,)

Everything here is identical to v0.2.5 — same data, same normalization, same BC behavior policy.
The only change is the **target policy scorer** (200demos_epoch40, ~90% SR) and the **oracle value**.

In [4]:
# Load the NEW target policy checkpoint (200demos_epoch40) for scoring
ckpt = load_checkpoint(CKPT_DIR_TARGET, ckpt_path=Path("models/model_epoch_40.pth"))

# Verify pre-collected rollouts exist (same 54% SR rollouts as v0.2.5)
rollout_paths = sorted(TARGET_ROLLOUT_DIR.glob("rollout_*.h5"))
assert len(rollout_paths) >= NUM_TARGET_ROLLOUTS, \
    f"Expected {NUM_TARGET_ROLLOUTS} rollouts in {TARGET_ROLLOUT_DIR}, found {len(rollout_paths)}"
rollout_paths = rollout_paths[:NUM_TARGET_ROLLOUTS]
print(f"Found {len(rollout_paths)} pre-collected rollouts in {TARGET_ROLLOUT_DIR}")
print(f"Target policy for scoring: 200demos_epoch40 from {CKPT_DIR_TARGET}")

Found 50 pre-collected rollouts in /home1/reishuen/latent_sope/rollouts/target_policy_50
Target policy for scoring: 200demos_epoch40 from /home1/reishuen/latent_sope/third_party/robomimic/diffusion_policy_trained_models/lift_diffusion_200demos/20260311141036


## Step 2: Load Target Rollouts + Expert Demos as Training Data

Load the collected `.h5` target rollouts AND the 200 expert demos from the robomimic HDF5.
Combine both into a single training set for the chunk diffuser, giving it a broad
distribution of trajectories (mixed quality).

In [5]:
# ── Part A: Load target policy rollouts (50 episodes from .h5 files) ──
target_data = []
all_states_list = []
all_actions_list = []

for path in tqdm(sorted(TARGET_ROLLOUT_DIR.glob("rollout_*.h5"))[:NUM_TARGET_ROLLOUTS],
                 desc="Loading target rollouts"):
    with h5py.File(path, "r") as f:
        latents = f["latents"][:]  # (T, frame_stack, D) or (T, D)
        actions = f["actions"][:]  # (T, Da)
        rewards = f["rewards"][:] if "rewards" in f else np.zeros(len(actions))
    
    if latents.ndim == 3:
        states = latents[:, -1, :]
    else:
        states = latents
    
    states = states.astype(np.float32)
    actions = actions.astype(np.float32)
    
    episode = {
        "states": states[:-1],
        "actions": actions[:-1],
        "rewards": rewards[:-1] if len(rewards) > 1 else rewards,
        "next-states": states[1:],
    }
    target_data.append(episode)
    all_states_list.append(states)
    all_actions_list.append(actions)

print(f"Loaded {len(target_data)} target policy episodes")

# ── Part B: Load expert demos (200 episodes from HDF5) ──
expert_data = []
with h5py.File(DEMO_HDF5, "r") as f:
    demo_keys = sorted(f["data"].keys(), key=lambda x: int(x.split("_")[1]))
    for dk in tqdm(demo_keys, desc="Loading expert demos"):
        demo = f[f"data/{dk}"]
        # Concatenate obs keys in sorted order to match LowDimConcatEncoder layout
        obs_arrays = [demo["obs"][k][:].astype(np.float32) for k in OBS_KEYS]
        states = np.concatenate(obs_arrays, axis=-1)  # (T, 19)
        actions = demo["actions"][:].astype(np.float32)  # (T, 7)
        rewards = demo["rewards"][:].astype(np.float32)
        
        episode = {
            "states": states[:-1],
            "actions": actions[:-1],
            "rewards": rewards[:-1],
            "next-states": states[1:],
        }
        expert_data.append(episode)
        all_states_list.append(states)
        all_actions_list.append(actions)

print(f"Loaded {len(expert_data)} expert demo episodes")

# ── Combine all data for normalization ──
all_episodes = target_data + expert_data
all_states = np.concatenate(all_states_list, axis=0)
all_actions = np.concatenate(all_actions_list, axis=0)
total_transitions = sum(len(ep["states"]) for ep in all_episodes)

# Compute normalization from ALL data (target + expert)
norm_mean = np.concatenate([all_states.mean(0), all_actions.mean(0)]).astype(np.float32)
norm_std = np.maximum(np.concatenate([all_states.std(0), all_actions.std(0)]), 1e-6).astype(np.float32)
norm_mean_t = torch.tensor(norm_mean, device=device)
norm_std_t = torch.tensor(norm_std, device=device)
normalize_fn = lambda x: (x - norm_mean_t) / norm_std_t
unnormalize_fn = lambda x: x * norm_std_t + norm_mean_t

# Compute success rates
target_successes = []
for ep in target_data:
    max_cube_z = ep["states"][:, CUBE_Z_INDEX].max()
    target_successes.append(max_cube_z > LIFT_THRESHOLD)
target_sr = np.mean(target_successes)

expert_successes = []
for ep in expert_data:
    max_cube_z = ep["states"][:, CUBE_Z_INDEX].max()
    expert_successes.append(max_cube_z > LIFT_THRESHOLD)
expert_sr = np.mean(expert_successes)

print(f"\nCombined: {len(all_episodes)} episodes ({len(target_data)} target + {len(expert_data)} expert), {total_transitions} transitions")
print(f"State dim: {all_states.shape[1]}, Action dim: {all_actions.shape[1]}")
print(f"Target policy SR (from rollouts): {target_sr*100:.1f}%")
print(f"Expert demo SR: {expert_sr*100:.1f}%")
print(f"Oracle SR: {oracle_success_rate*100:.1f}%")
print(f"Norm mean shape: {norm_mean.shape}")

Loading target rollouts:   0%|          | 0/50 [00:00<?, ?it/s]

Loading target rollouts:  58%|█████▊    | 29/50 [00:00<00:00, 281.72it/s]

Loading target rollouts: 100%|██████████| 50/50 [00:00<00:00, 293.50it/s]

Loaded 50 target policy episodes


Loading expert demos:   0%|          | 0/200 [00:00<?, ?it/s]

...

Loading expert demos:  57%|█████▋    | 114/200 [00:00<00:00, 269.47it/s]

Loading expert demos:  71%|███████   | 142/200 [00:00<00:00, 272.05it/s]

Loading expert demos:  85%|████████▌ | 170/200 [00:00<00:00, 254.34it/s]

Loading expert demos: 100%|██████████| 200/200 [00:00<00:00, 274.11it/s]

Loaded 200 expert demo episodes

Combined: 250 episodes (50 target + 200 expert), 12091 transitions
State dim: 19, Action dim: 7
Target policy SR (from rollouts): 0.0%
Expert demo SR: 100.0%
Oracle SR: 90.0%
Norm mean shape: (26,)


## Step 3: Chunk Target Rollouts for Diffusion Training

In [6]:
# Chunk ALL episodes (target + expert) into training data
chunks_states_from = []
chunks_actions_from = []
chunks_states_to = []
chunks_actions_to = []

for ep in all_episodes:
    states = ep["states"]   # (T, 19)
    actions = ep["actions"]  # (T, 7)
    T = len(states)
    
    if T < CHUNK_SIZE + 1:
        continue
    
    for t0 in range(0, T - CHUNK_SIZE, STRIDE):
        s_to = states[t0 : t0 + CHUNK_SIZE + 1]  # (5, 19)
        a_to = actions[t0 : t0 + CHUNK_SIZE]      # (4, 7)
        
        if t0 > 0:
            s_from = states[t0 - 1 : t0].reshape(1, STATE_DIM)
            a_from = actions[t0 - 1 : t0].reshape(1, ACTION_DIM)
        else:
            s_from = states[0:1].reshape(1, STATE_DIM)
            a_from = actions[0:1].reshape(1, ACTION_DIM)
        
        chunks_states_from.append(s_from)
        chunks_actions_from.append(a_from)
        chunks_states_to.append(s_to)
        chunks_actions_to.append(a_to)

chunks_states_from = np.array(chunks_states_from, dtype=np.float32)
chunks_actions_from = np.array(chunks_actions_from, dtype=np.float32)
chunks_states_to = np.array(chunks_states_to, dtype=np.float32)
chunks_actions_to = np.array(chunks_actions_to, dtype=np.float32)

print(f"Chunked {len(all_episodes)} episodes ({len(target_data)} target + {len(expert_data)} expert) into {len(chunks_states_to)} training chunks")
print(f"  states_from: {chunks_states_from.shape}")
print(f"  actions_from: {chunks_actions_from.shape}")
print(f"  states_to: {chunks_states_to.shape}")
print(f"  actions_to: {chunks_actions_to.shape}")
print(f"  Batches per epoch: {len(chunks_states_to) // BATCH_SIZE}")

Chunked 250 episodes (50 target + 200 expert) into 5611 training chunks
  states_from: (5611, 1, 19)
  actions_from: (5611, 1, 7)
  states_to: (5611, 5, 19)
  actions_to: (5611, 4, 7)
  Batches per epoch: 87


## Step 4: Load Pre-trained Chunk Diffuser from v0.2.5

Reuse the chunk diffuser trained in v0.2.5 on mixed data (50 target rollouts ~54% SR + 200 expert demos).
No retraining — the diffuser captures the general trajectory distribution.

In [7]:
# Build TemporalUnet + GaussianDiffusion (same architecture as v0.2.5)
temporal_model = TemporalUnet(
    horizon=CHUNK_SIZE,
    transition_dim=TRANSITION_DIM,
    dim=BASE_DIM,
    dim_mults=DIM_MULTS,
    attention=False,
).to(device)

diffusion_model = GaussianDiffusion(
    model=temporal_model,
    horizon=CHUNK_SIZE,
    observation_dim=STATE_DIM,
    action_dim=ACTION_DIM,
    n_timesteps=N_DIFFUSION_STEPS,
    normalizer=normalize_fn,
    unnormalizer=unnormalize_fn,
    predict_epsilon=PREDICT_EPSILON,
    loss_type="l2",
    clip_denoised=False,
    action_weight=ACTION_WEIGHT,
    loss_weights=None,
    loss_discount=1.0,
).to(device)

ema = EMA(diffusion_model)

# Load pre-trained checkpoint from v0.2.5
V025_CKPT_DIR = PROJECT_ROOT / "diffusion_ckpts" / "mvp_v025_reward_fix"
diffusion_model.load_state_dict(torch.load(V025_CKPT_DIR / "diffusion_model.pt", map_location=device))
ema.ema_model.load_state_dict(torch.load(V025_CKPT_DIR / "diffusion_model_ema.pt", map_location=device))

n_params = sum(p.numel() for p in diffusion_model.parameters())
print(f"Loaded pre-trained diffusion model from v0.2.5: {n_params:,} parameters")
print(f"Checkpoint: {V025_CKPT_DIR}")

/home1/reishuen/latent_sope/third_party/sope/opelab/core/baselines/diffusion/diffusion.py:314: DeprecationWarning: __array_wrap__ must accept context and return_scalar arguments (positionally) in the future. (Deprecated NumPy 2.0)
  betas * np.sqrt(alphas_cumprod_prev) / (1. - alphas_cumprod))


[ models/temporal ] Channel dimensions: [(26, 32), (32, 128), (128, 256)]
[(26, 32), (32, 128), (128, 256)]


Loaded pre-trained diffusion model from v0.2.5: 3,686,618 parameters
Checkpoint: /home1/reishuen/latent_sope/diffusion_ckpts/mvp_v025_reward_fix


In [8]:
# No training needed — using pre-trained v0.2.5 diffuser
# Set to eval mode
diffusion_model.eval()
ema.ema_model.eval()

# Create empty epoch_losses for compatibility with visualization cells
epoch_losses = []
print("Skipping training — using pre-trained v0.2.5 chunk diffuser")

Skipping training — using pre-trained v0.2.5 chunk diffuser


## Step 5: Train BC_Gaussian Behavior Policy

SOPE always uses a Gaussian MLP as the behavior policy for negative guidance,
even when the target policy is a diffusion model. Train on the **same target
policy rollout data** (this represents the "behavior" distribution that the
chunk diffuser was trained on).

Unlike v0.2.3 which used a diffusion behavior policy (noisy scores),
this uses analytical Gaussian gradients: `grad = -(a - mu(s)) / sigma^2`.

In [9]:
class BCGaussian(nn.Module):
    """Simple BC policy with Gaussian output: pi(a|s) = N(mu(s), sigma(s)).
    
    Provides analytical grad_log_prob for SOPE-style guidance.
    """
    def __init__(self, state_dim, action_dim, hidden_dim=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
        )
        self.mean_head = nn.Linear(hidden_dim, action_dim)
        self.log_std_head = nn.Linear(hidden_dim, action_dim)
    
    def forward(self, state):
        h = self.net(state)
        mean = self.mean_head(h)
        log_std = self.log_std_head(h).clamp(-5, 2)
        return mean, log_std
    
    def log_prob(self, state, action):
        mean, log_std = self.forward(state)
        std = torch.exp(log_std)
        return -0.5 * (((action - mean) / std) ** 2 + 2 * log_std + math.log(2 * math.pi)).sum(dim=-1)
    
    def grad_log_prob(self, state, action):
        """Analytical gradient: d/da log N(a | mu(s), sigma(s)) = -(a - mu) / sigma^2."""
        with torch.no_grad():
            mean, log_std = self.forward(state)
            std = torch.exp(log_std)
            return -(action - mean) / (std ** 2)
    
    def grad_log_prob_chunk(self, states, actions):
        """Chunk-level scoring: (B, T, state_dim), (B, T, action_dim) -> (B, T, action_dim)."""
        B, T, _ = states.shape
        s_flat = states.reshape(B * T, -1)
        a_flat = actions.reshape(B * T, -1)
        grad_flat = self.grad_log_prob(s_flat, a_flat)
        return grad_flat.reshape(B, T, -1)


# Train on target policy rollout (state, action) pairs
demo_states = np.concatenate([ep["states"] for ep in target_data], axis=0)
demo_actions = np.concatenate([ep["actions"] for ep in target_data], axis=0)
print(f"Behavior policy training data: {demo_states.shape[0]} (state, action) pairs")

bc_behavior = BCGaussian(STATE_DIM, ACTION_DIM, hidden_dim=256).to(device)
bc_optimizer = torch.optim.Adam(bc_behavior.parameters(), lr=1e-3)

states_t = torch.tensor(demo_states, dtype=torch.float32, device=device)
actions_t = torch.tensor(demo_actions, dtype=torch.float32, device=device)

BC_EPOCHS = 500
BC_BATCH = 256
bc_losses = []

print(f"Training BC_Gaussian behavior policy for {BC_EPOCHS} epochs...")
bc_behavior.train()
for epoch in range(BC_EPOCHS):
    idx = torch.randint(0, len(states_t), (BC_BATCH,), device=device)
    nll = -bc_behavior.log_prob(states_t[idx], actions_t[idx]).mean()
    bc_optimizer.zero_grad()
    nll.backward()
    bc_optimizer.step()
    bc_losses.append(nll.item())

bc_behavior.eval()
print(f"Final NLL: {bc_losses[-1]:.4f}")

# Sanity check
test_s = states_t[:8]
test_a = actions_t[:8]
test_grad = bc_behavior.grad_log_prob(test_s, test_a)
print(f"Behavior grad norms: {test_grad.norm(dim=-1).cpu().numpy()}")

Behavior policy training data: 2625 (state, action) pairs
Training BC_Gaussian behavior policy for 500 epochs...


Final NLL: -7.0053


Behavior grad norms: [88.227036 42.753716 52.82374  34.68356  23.762173 19.691381 27.685867
 28.395767]


## Step 6: Load Target Policy Scorer (200demos_epoch40)

This is the key change from v0.2.5: we load the **200demos_epoch40** diffusion policy
(~90% SR) as the target scorer, instead of the original test checkpoint (~54% SR).

In [10]:
target_algo = build_algo_from_checkpoint(ckpt, device=str(device))
target_scorer = RobomimicDiffusionScorer(target_algo, device=str(device), obs_keys=OBS_KEYS)
print(f"Target scorer: state_dim={target_scorer.state_dim}, action_dim={target_scorer.action_dim}")
print(f"  sigma[1]={target_scorer.sigma:.6f}, prediction_horizon={target_scorer.prediction_horizon}")

# Compare target vs behavior score magnitudes
test_s = states_t[:8]
test_a = actions_t[:8]
target_score = target_scorer.grad_log_prob(test_s, test_a)
behavior_score = bc_behavior.grad_log_prob(test_s, test_a)
print(f"\nTarget score norms:   {target_score.norm(dim=-1).cpu().numpy()}")
print(f"Behavior score norms: {behavior_score.norm(dim=-1).cpu().numpy()}")
print(f"Target/Behavior norm ratio: {(target_score.norm(dim=-1) / (behavior_score.norm(dim=-1) + 1e-8)).mean().item():.2f}")


============= Initialized Observation Utils with Obs Spec =============

using obs modality: low_dim with keys: ['robot0_eef_pos', 'robot0_gripper_qpos', 'robot0_eef_quat', 'object']
using obs modality: rgb with keys: []
using obs modality: depth with keys: []
using obs modality: scan with keys: []


number of parameters: 6.576359e+07


[10:56:03] INFO     build_algo_from_checkpoint took 0.48 seconds to execute                           ]8;id=87611;file:///home1/reishuen/latent_sope/src/latent_sope/utils/common.py\common.py]8;;\:]8;id=821403;file:///home1/reishuen/latent_sope/src/latent_sope/utils/common.py#137\137]8;;\

Target scorer: state_dim=19, action_dim=7
  sigma[1]=0.041803, prediction_horizon=16



Target score norms:   [26.768578 19.36106  16.333904 14.13505  16.352375 13.824146 17.966135
 15.817644]
Behavior score norms: [88.227036 42.753716 52.82374  34.68356  23.762173 19.691381 27.685867
 28.395767]
Target/Behavior norm ratio: 0.51


## Step 7: Guided Stitching with Corrected SOPE Hyperparameters + Fixed Reward

Key changes from v0.2.3:
- `clamp=False` (SOPE disables clamping for diffusion targets)
- `k_guide=1` (not 2)
- `use_adaptive=False` (no cosine decay)

**v0.2.5 fix:** `score_trajectories_gt` now uses **episode-level binary reward**
(1.0 if cube lifted at any step, 0.0 otherwise) matching robosuite's `total_reward`.
Previously summed per-step rewards, giving returns in [0, 60] vs oracle in [0, 1].

In [11]:
def get_schedule_multiplier(t, n_timesteps):
    """Cosine schedule multiplier for adaptive guidance scaling."""
    t_frac = t / n_timesteps
    return 0.5 * (1 + math.cos(math.pi * t_frac))


def get_initial_states_from_data(data, num_samples, device):
    """Sample initial states from the dataset."""
    all_initial = np.array([ep["states"][0] for ep in data])
    indices = np.random.choice(len(all_initial), num_samples, replace=True)
    return torch.tensor(all_initial[indices], dtype=torch.float32, device=device)


def generate_trajectories_full_guidance(
    diffusion_model, initial_states,
    normalize_fn, unnormalize_fn,
    state_dim, action_dim,
    chunk_size, t_gen, device,
    target_scorer=None, behavior_scorer=None,
    action_scale=0.0, ratio=0.5,
    k_guide=1, normalize_grad=True,
    use_adaptive=False, clamp=False, l_inf=1.0,
):
    """Generate trajectories with full SOPE-style guidance (positive + negative).
    
    Matches SOPE's default_sample_fn() with corrected settings for diffusion targets:
    - clamp=False, use_adaptive=False, k_guide=1
    """
    guided = (target_scorer is not None and action_scale > 0)
    use_neg_grad = (behavior_scorer is not None and ratio > 0)
    batch_size = initial_states.shape[0]
    transition_dim = state_dim + action_dim
    n_timesteps = diffusion_model.n_timesteps

    # Normalize initial states for conditioning
    padded = torch.cat([
        initial_states,
        torch.zeros(batch_size, action_dim, device=device)
    ], dim=1)
    normalized_initial = normalize_fn(padded)[:, :state_dim]

    all_trajectories = torch.zeros(batch_size, t_gen, transition_dim, device=device)
    conditions = {0: normalized_initial}
    total_generated = 0
    n_iterations = 0

    while total_generated < t_gen:
        n_iterations += 1
        steps_remaining = t_gen - total_generated
        shape = (batch_size, chunk_size, transition_dim)

        # Initialize from noise
        x = torch.randn(shape, device=device)
        x = apply_conditioning(x, conditions, state_dim)

        for t_diff in reversed(range(n_timesteps)):
            t_tensor = torch.full((batch_size,), t_diff, device=device, dtype=torch.long)

            with torch.no_grad():
                model_mean, _, model_log_variance = diffusion_model.p_mean_variance(x=x, t=t_tensor)
                model_std = torch.exp(0.5 * model_log_variance)

            if guided:
                # Unnormalize to real space (guidance operates in original data space)
                model_mean = unnormalize_fn(model_mean)

                for _k in range(k_guide):
                    # Extract states and actions from current mean
                    states_chunk = model_mean[:, :, :state_dim]
                    actions_chunk = model_mean[:, :, state_dim:]

                    # Positive: target policy score
                    target_grad = target_scorer.grad_log_prob_chunk(states_chunk, actions_chunk)

                    # Negative: behavior policy score
                    if use_neg_grad:
                        behavior_grad = behavior_scorer.grad_log_prob_chunk(states_chunk, actions_chunk)

                    # Per-timestep L2 normalization
                    if normalize_grad:
                        eps = 1e-6
                        target_norm = target_grad.norm(dim=-1, keepdim=True) + eps
                        target_grad = target_grad / target_norm
                        if use_neg_grad:
                            behavior_norm = behavior_grad.norm(dim=-1, keepdim=True) + eps
                            behavior_grad = behavior_grad / behavior_norm

                    # Clamp (disabled for diffusion policies per SOPE)
                    if clamp:
                        target_grad = target_grad.clamp(-l_inf, l_inf)
                        if use_neg_grad:
                            behavior_grad = behavior_grad.clamp(-l_inf, l_inf)

                    # Combine: guide = target - ratio * behavior
                    if use_neg_grad:
                        guide_actions = target_grad - ratio * behavior_grad
                    else:
                        guide_actions = target_grad

                    # Build full guide tensor (zeros for states)
                    guide = torch.zeros_like(model_mean)
                    guide[:, :, state_dim:] = guide_actions

                    # Scale
                    if use_adaptive:
                        scale_mult = get_schedule_multiplier(n_timesteps - t_diff, n_timesteps)
                        guide = scale_mult * action_scale * guide
                    else:
                        guide = action_scale * guide

                    # Apply guidance
                    model_mean = model_mean + guide

                    # Re-normalize, re-condition, un-normalize
                    model_mean = normalize_fn(model_mean)
                    model_mean = apply_conditioning(model_mean, conditions, state_dim)
                    model_mean = unnormalize_fn(model_mean)

                # Final re-normalize before adding noise
                model_mean = normalize_fn(model_mean)

            # Add noise (standard diffusion sampling)
            noise = torch.randn_like(x)
            nonzero_mask = (1 - (t_diff == 0) * 1.0)
            x = model_mean + nonzero_mask * model_std * noise
            x = apply_conditioning(x, conditions, state_dim)

        # Unnormalize the generated chunk
        chunk_unnorm = unnormalize_fn(x)

        # Store chunk (1-step overlap)
        steps_to_store = min(chunk_size - 1, steps_remaining)
        all_trajectories[:, total_generated:total_generated + steps_to_store] = chunk_unnorm[:, :steps_to_store]
        total_generated += steps_to_store

        if total_generated >= t_gen:
            break

        # Condition next chunk on last state (stay in normalized space)
        last_states_norm = x[:, -1, :state_dim]
        conditions = {0: last_states_norm}

    label = "unguided"
    if guided and use_neg_grad:
        label = f"full(s={action_scale},r={ratio})"
    elif guided:
        label = f"pos_only(s={action_scale})"
    print(f"  [{label}] Generated {total_generated} steps in {n_iterations} chunks")
    return all_trajectories.detach().cpu().numpy()


def score_trajectories_gt(trajectories, cube_z_index, threshold):
    """Score trajectories using robosuite's episode-level Lift reward.
    
    Matches robosuite's Lift environment: reward = 1.0 if cube_z > threshold
    at ANY timestep during the episode, 0.0 otherwise.
    
    v0.2.5 FIX: Previously this summed per-step binary rewards (range 0-60),
    but oracle uses episode-level binary reward (range 0-1). Now both match.
    
    Args:
        trajectories: (B, T, D) array of generated trajectories.
        cube_z_index: index of cube z-position in state vector.
        threshold: cube z threshold for success (0.84 for Lift).
    
    Returns:
        returns: (B,) array — 1.0 if episode succeeded, 0.0 otherwise.
        successes: (B,) bool array — True if cube lifted at any timestep.
    """
    B, T, D = trajectories.shape
    # Episode succeeds if cube_z exceeds threshold at ANY timestep
    cube_z = trajectories[:, :, cube_z_index]  # (B, T)
    successes = np.any(cube_z > threshold, axis=1)  # (B,)
    returns = successes.astype(np.float32)  # 1.0 or 0.0
    return returns, successes


print("Generation and scoring functions defined.")
print(f"NOTE: score_trajectories_gt uses episode-level binary reward (matching robosuite oracle)")

Generation and scoring functions defined.
NOTE: score_trajectories_gt uses episode-level binary reward (matching robosuite oracle)


## Step 8: Run Guidance Sweep

In [12]:
np.random.seed(42)
torch.manual_seed(42)

# Sample initial states from TARGET data (not expert demos)
initial_states = get_initial_states_from_data(target_data, NUM_SYNTHETIC_TRAJS, device)
results_by_config = {}

for i, gc in enumerate(GUIDANCE_CONFIGS):
    label = gc["label"]
    action_scale = gc["action_scale"]
    ratio = gc["ratio"]

    print("=" * 60)
    print(f"[{i+1}/{len(GUIDANCE_CONFIGS)}] {label}")
    print("=" * 60)

    t0 = time.time()

    trajs = generate_trajectories_full_guidance(
        diffusion_model=ema.ema_model,
        initial_states=initial_states,
        normalize_fn=normalize_fn, unnormalize_fn=unnormalize_fn,
        state_dim=STATE_DIM, action_dim=ACTION_DIM,
        chunk_size=CHUNK_SIZE, t_gen=T_GEN, device=device,
        target_scorer=target_scorer if action_scale > 0 else None,
        behavior_scorer=bc_behavior if (action_scale > 0 and ratio > 0) else None,
        action_scale=action_scale, ratio=ratio,
        k_guide=K_GUIDE, normalize_grad=NORMALIZE_GRAD,
        use_adaptive=USE_ADAPTIVE, clamp=CLAMP, l_inf=L_INF,
    )

    elapsed = time.time() - t0
    # v0.2.5: episode-level binary reward (no gamma parameter)
    returns, successes = score_trajectories_gt(trajs, CUBE_Z_INDEX, LIFT_THRESHOLD)

    rel_err = abs(np.mean(returns) - oracle_value) / abs(oracle_value) if oracle_value != 0 else float('inf')

    results_by_config[label] = {
        "trajs": trajs,
        "returns": returns,
        "successes": successes,
        "estimate": float(np.mean(returns)),
        "std": float(np.std(returns)),
        "success_rate": float(np.mean(successes)),
        "rel_error": rel_err,
        "action_scale": action_scale,
        "ratio": ratio,
        "time": elapsed,
    }
    print(f"  OPE={np.mean(returns):.4f}, SR={np.mean(successes)*100:.1f}%, "
          f"rel_err={rel_err:.2%}, time={elapsed:.0f}s\n")

# Summary table
print("=" * 70)
print("GUIDANCE SWEEP SUMMARY -- v0.2.5 (Reward Model Fix)")
print("=" * 70)
print(f"Oracle V^pi = {oracle_value:.4f} (SR = {oracle_success_rate*100:.1f}%)")
print(f"Target policy SR (from rollouts): {target_sr*100:.1f}%")
print(f"NOTE: OPE estimate = synthetic success rate (episode-level binary reward)")
print(f"\n{'Config':<22} {'Scale':>6} {'Ratio':>6} {'OPE':>8} {'SR':>6} {'RelErr':>10}")
print("-" * 62)
for label, r in results_by_config.items():
    print(f"{label:<22} {r['action_scale']:>6.2f} {r['ratio']:>6.2f} "
          f"{r['estimate']:>8.4f} {r['success_rate']*100:>5.1f}% {r['rel_error']:>9.2%}")

[1/10] unguided


  [unguided] Generated 60 steps in 20 chunks
  OPE=0.7600, SR=76.0%, rel_err=15.56%, time=49s

[2/10] pos_only_0.05


  [pos_only(s=0.05)] Generated 60 steps in 20 chunks
  OPE=0.6400, SR=64.0%, rel_err=28.89%, time=120s

[3/10] pos_only_0.1


  [pos_only(s=0.1)] Generated 60 steps in 20 chunks
  OPE=0.4800, SR=48.0%, rel_err=46.67%, time=120s

[4/10] pos_only_0.2


  [pos_only(s=0.2)] Generated 60 steps in 20 chunks
  OPE=0.3000, SR=30.0%, rel_err=66.67%, time=120s

[5/10] full_0.05_r0.25


...

  [full(s=0.1,r=0.25)] Generated 60 steps in 20 chunks
  OPE=0.4600, SR=46.0%, rel_err=48.89%, time=120s

[7/10] full_0.2_r0.25


  [full(s=0.2,r=0.25)] Generated 60 steps in 20 chunks
  OPE=0.6600, SR=66.0%, rel_err=26.67%, time=120s

[8/10] full_0.2_r0.5


  [full(s=0.2,r=0.5)] Generated 60 steps in 20 chunks
  OPE=0.0800, SR=8.0%, rel_err=91.11%, time=120s

[9/10] full_0.5_r0.25


  [full(s=0.5,r=0.25)] Generated 60 steps in 20 chunks
  OPE=0.7000, SR=70.0%, rel_err=22.22%, time=120s

[10/10] full_0.5_r0.5


  [full(s=0.5,r=0.5)] Generated 60 steps in 20 chunks
  OPE=0.0000, SR=0.0%, rel_err=100.00%, time=120s

GUIDANCE SWEEP SUMMARY -- v0.2.5 (Reward Model Fix)
Oracle V^pi = 0.9000 (SR = 90.0%)
Target policy SR (from rollouts): 0.0%
NOTE: OPE estimate = synthetic success rate (episode-level binary reward)

Config                  Scale  Ratio      OPE     SR     RelErr
--------------------------------------------------------------
unguided                 0.00   0.00   0.7600  76.0%    15.56%
pos_only_0.05            0.05   0.00   0.6400  64.0%    28.89%
pos_only_0.1             0.10   0.00   0.4800  48.0%    46.67%
pos_only_0.2             0.20   0.00   0.3000  30.0%    66.67%
full_0.05_r0.25          0.05   0.25   0.4600  46.0%    48.89%
full_0.1_r0.25           0.10   0.25   0.4600  46.0%    48.89%
full_0.2_r0.25           0.20   0.25   0.6600  66.0%    26.67%
full_0.2_r0.5            0.20   0.50   0.0800   8.0%    91.11%
full_0.5_r0.25           0.50   0.25   0.7000  70.0%    22.22%
f

## Step 9: Visualization

In [13]:
# ── Find best config ──
best_label = min(results_by_config, key=lambda k: results_by_config[k]["rel_error"])
best_r = results_by_config[best_label]
print(f"Best config: {best_label} (rel_error={best_r['rel_error']:.2%})")

# ── Build real target trajectories for comparison ──
real_trajs = []
for ep in target_data[:20]:
    traj = np.concatenate([ep["states"], ep["actions"]], axis=-1)
    if len(traj) >= T_GEN:
        real_trajs.append(traj[:T_GEN])
    else:
        padded = np.zeros((T_GEN, TRANSITION_DIM), dtype=np.float32)
        padded[:len(traj)] = traj
        padded[len(traj):] = traj[-1]
        real_trajs.append(padded)
real_trajs = np.array(real_trajs)

unguided_trajs = results_by_config["unguided"]["trajs"]
best_trajs = best_r["trajs"]

# ── Figure 1: OPE + Success Rate summary ──
fig, axes = plt.subplots(1, 3, figsize=(20, 5))

labels = list(results_by_config.keys())
estimates = [results_by_config[l]["estimate"] for l in labels]
stds = [results_by_config[l]["std"] for l in labels]
success_rates = [results_by_config[l]["success_rate"] * 100 for l in labels]

colors = []
for l in labels:
    if "unguided" in l:
        colors.append("steelblue")
    elif "pos_only" in l:
        colors.append("orange")
    else:
        colors.append("coral")

# Panel 1: OPE estimates (now in [0, 1] range!)
axes[0].bar(range(len(labels)), estimates, color=colors, edgecolor="black")
axes[0].errorbar(range(len(labels)), estimates, yerr=stds, fmt="none", color="black", capsize=3)
axes[0].axhline(y=oracle_value, color="green", linestyle="--", linewidth=2, label=f"Oracle={oracle_value:.2f}")
axes[0].set_xticks(range(len(labels)))
axes[0].set_xticklabels(labels, rotation=60, ha="right", fontsize=7)
axes[0].set_ylabel("OPE Estimate (= Success Rate)")
axes[0].set_ylim([0, 1.05])
axes[0].set_title("OPE by Guidance Config (episode-level reward)")
axes[0].legend(fontsize=8)
axes[0].grid(True, alpha=0.3, axis="y")

# Panel 2: Success rates (now identical to OPE since reward is binary)
axes[1].bar(range(len(labels)), success_rates, color=colors, edgecolor="black")
axes[1].axhline(y=oracle_success_rate * 100, color="green", linestyle="--", linewidth=2, label=f"Oracle={oracle_success_rate*100:.0f}%")
axes[1].set_xticks(range(len(labels)))
axes[1].set_xticklabels(labels, rotation=60, ha="right", fontsize=7)
axes[1].set_ylabel("Success Rate (%)")
axes[1].set_title("Synthetic Success Rate")
axes[1].legend(fontsize=8)
axes[1].grid(True, alpha=0.3, axis="y")

# Panel 3: Cube z for best config vs unguided vs real
for j in range(min(10, len(real_trajs))):
    axes[2].plot(real_trajs[j, :, CUBE_Z_INDEX], alpha=0.15, color="green",
                 label="Target real" if j == 0 else "")
for j in range(min(10, NUM_SYNTHETIC_TRAJS)):
    axes[2].plot(unguided_trajs[j, :, CUBE_Z_INDEX], alpha=0.2, color="steelblue",
                 label="Unguided" if j == 0 else "")
    axes[2].plot(best_trajs[j, :, CUBE_Z_INDEX], alpha=0.2, color="coral",
                 label=f"Best ({best_label})" if j == 0 else "")
axes[2].axhline(y=LIFT_THRESHOLD, color="red", linestyle=":", alpha=0.5, label="Lift threshold")
axes[2].set_xlabel("Timestep")
axes[2].set_ylabel("Cube z")
axes[2].set_title("Trajectory Quality")
axes[2].legend(fontsize=7)
axes[2].grid(True, alpha=0.3)

plt.suptitle("MVP v0.2.5: Reward Model Fix (Episode-Level Binary Reward)", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

Best config: unguided (rel_error=15.56%)


In [14]:
# ── Figure 2: Per-dimension trajectory comparison (real vs unguided vs best) ──
KEY_DIMS = {
    "cube_z": 2, "cube_x": 0, "eef_z": 12, "eef_x": 10,
    "grip2cube_z": 9, "gripper_q0": 17,
}

N_SHOW = 10
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for idx, (name, dim) in enumerate(KEY_DIMS.items()):
    ax = axes[idx]
    for j in range(min(N_SHOW, len(real_trajs))):
        ax.plot(real_trajs[j, :, dim], color="green", alpha=0.15,
                label="Target real" if j == 0 else "")
    for j in range(min(N_SHOW, NUM_SYNTHETIC_TRAJS)):
        ax.plot(unguided_trajs[j, :, dim], color="steelblue", alpha=0.25,
                label="Unguided" if j == 0 else "")
    for j in range(min(N_SHOW, NUM_SYNTHETIC_TRAJS)):
        ax.plot(best_trajs[j, :, dim], color="coral", alpha=0.25,
                label=f"Best ({best_label})" if j == 0 else "")
    if dim == CUBE_Z_INDEX:
        ax.axhline(y=LIFT_THRESHOLD, color="red", linestyle=":", alpha=0.5)
    ax.set_title(name, fontsize=11)
    ax.set_xlabel("Timestep")
    ax.grid(True, alpha=0.3)
    if idx == 0:
        ax.legend(fontsize=7, loc="upper left")

plt.suptitle("v0.2.5: State Trajectories — Target Real (green) vs Unguided (blue) vs Best Guided (red)",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

In [15]:
# ── Figure 3: Cube z across ALL configs ──
n_configs = len(results_by_config)
n_cols = 4
n_rows = math.ceil(n_configs / n_cols)
fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 4 * n_rows))
axes = axes.flatten()

for idx, (label, r) in enumerate(results_by_config.items()):
    ax = axes[idx]
    trajs = r["trajs"]
    for j in range(min(5, len(real_trajs))):
        ax.plot(real_trajs[j, :, CUBE_Z_INDEX], color="green", alpha=0.1)
    for j in range(min(15, NUM_SYNTHETIC_TRAJS)):
        ax.plot(trajs[j, :, CUBE_Z_INDEX], color="coral", alpha=0.2)
    ax.plot(trajs[:, :, CUBE_Z_INDEX].mean(axis=0), color="darkred", linewidth=2, label="Mean")
    ax.plot(real_trajs[:, :, CUBE_Z_INDEX].mean(axis=0), color="darkgreen", linewidth=2,
            linestyle="--", label="Real mean")
    ax.axhline(y=LIFT_THRESHOLD, color="red", linestyle=":", alpha=0.5)
    ax.set_title(f"{label}\nOPE={r['estimate']:.2f}, SR={r['success_rate']*100:.0f}%", fontsize=9)
    ax.set_xlabel("Timestep")
    ax.set_ylabel("cube_z")
    ax.set_ylim([0.78, 0.95])
    ax.grid(True, alpha=0.3)
    if idx == 0:
        ax.legend(fontsize=7)

for idx in range(n_configs, len(axes)):
    axes[idx].set_visible(False)

plt.suptitle("v0.2.5: Cube Z Across All Guidance Configs (Episode-Level Reward)", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

In [16]:
# ── Figure 3b: Individual trajectory visualizations (5 synthetic vs 5 real) ──
# Show full state trajectories for individual episodes to debug trajectory quality
N_INDIV = 5
KEY_STATE_DIMS = {
    "cube_x": 0, "cube_y": 1, "cube_z": 2,
    "eef_x": 10, "eef_y": 11, "eef_z": 12,
}

fig, axes = plt.subplots(N_INDIV, len(KEY_STATE_DIMS), figsize=(3.5 * len(KEY_STATE_DIMS), 3 * N_INDIV))

for row in range(N_INDIV):
    for col, (dim_name, dim_idx) in enumerate(KEY_STATE_DIMS.items()):
        ax = axes[row, col]
        # Real trajectory
        if row < len(real_trajs):
            ax.plot(real_trajs[row, :, dim_idx], color="green", linewidth=1.5, label="Real" if col == 0 else "")
        # Best guided synthetic
        ax.plot(best_trajs[row, :, dim_idx], color="coral", linewidth=1.5, linestyle="--",
                label=f"Synth ({best_label})" if col == 0 else "")
        # Unguided synthetic
        ax.plot(unguided_trajs[row, :, dim_idx], color="steelblue", linewidth=1.0, linestyle=":",
                alpha=0.7, label="Unguided" if col == 0 else "")
        
        if dim_idx == CUBE_Z_INDEX:
            ax.axhline(y=LIFT_THRESHOLD, color="red", linestyle=":", alpha=0.4)
        
        if row == 0:
            ax.set_title(dim_name, fontsize=10, fontweight="bold")
        if col == 0:
            real_succ = "S" if (row < len(real_trajs) and np.any(real_trajs[row, :, CUBE_Z_INDEX] > LIFT_THRESHOLD)) else "F"
            synth_succ = "S" if best_r["successes"][row] else "F"
            ax.set_ylabel(f"Traj {row}\nreal={real_succ} synth={synth_succ}", fontsize=8)
        ax.grid(True, alpha=0.2)
        ax.tick_params(labelsize=7)
        
        if row == 0 and col == 0:
            ax.legend(fontsize=6, loc="upper left")

plt.suptitle(f"v0.2.5: Individual Trajectories — Real (green) vs Best Synthetic (red) vs Unguided (blue)",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

In [17]:
# ── Figure 4: Training curves ──
# Note: chunk diffuser was loaded from v0.2.5 (no training), so only show BC training

if epoch_losses:
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    axes[0].plot(epoch_losses, marker='o', markersize=3)
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Loss")
    axes[0].set_title(f"Chunk Diffuser Training ({TRAIN_EPOCHS} epochs, target+expert data)")
    axes[0].grid(True, alpha=0.3)
    ax_bc = axes[1]
else:
    fig, ax_bc = plt.subplots(1, 1, figsize=(7, 4))

ax_bc.plot(bc_losses, alpha=0.5)
ax_bc.set_xlabel("Epoch")
ax_bc.set_ylabel("NLL")
ax_bc.set_title(f"BC_Gaussian Behavior Policy ({BC_EPOCHS} epochs)")
ax_bc.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Step 10: Summary

In [18]:
print(f"\n{'='*60}")
print("MVP v0.2.5.1 COMPLETE — 90% Target Policy (200demos_epoch40)")
print(f"{'='*60}")
print(f"Best: {best_label}, OPE={best_r['estimate']:.4f}, rel_error={best_r['rel_error']:.2%}")
print(f"Oracle: V^pi={oracle_value:.4f}")
print(f"\nSetup:")
print(f"  Target policy: 200demos_epoch40 (~90% SR)")
print(f"  Chunk diffuser: pre-trained from v0.2.5 (54% SR target + expert data)")
print(f"  Reward: episode-level binary (1.0 if cube_z > {LIFT_THRESHOLD} at any step)")
print(f"\nData: {len(target_data)} target rollouts + {len(expert_data)} expert demos = {len(all_episodes)} episodes")
print(f"Target SR: {target_sr*100:.1f}%, Expert SR: {expert_sr*100:.1f}%")
print(f"\nComparison:")
print(f"  v0.2.5 (54% SR target, same diffuser): best rel_error from prior run")
print(f"  v0.2.5.1 (90% SR target, same diffuser): best rel_error={best_r['rel_error']:.2%}")


MVP v0.2.5.1 COMPLETE — 90% Target Policy (200demos_epoch40)
Best: unguided, OPE=0.7600, rel_error=15.56%
Oracle: V^pi=0.9000

Setup:
  Target policy: 200demos_epoch40 (~90% SR)
  Chunk diffuser: pre-trained from v0.2.5 (54% SR target + expert data)
  Reward: episode-level binary (1.0 if cube_z > 0.84 at any step)

Data: 50 target rollouts + 200 expert demos = 250 episodes
Target SR: 0.0%, Expert SR: 100.0%

Comparison:
  v0.2.5 (54% SR target, same diffuser): best rel_error from prior run
  v0.2.5.1 (90% SR target, same diffuser): best rel_error=15.56%
